# 13 · How we build /api/llm/generate (A2 / A3 / A4 shared endpoint)

**Powers**:
- [A2 文創敘事生成器](https://lius.cc/llm/demos/cultural-narrative)
- [A3 出版選題雷達](https://lius.cc/llm/demos/publishing-radar)
- [A4 文化部標案副駕](https://lius.cc/llm/demos/gov-grant-copilot)

**Idea**: one endpoint, three task-specific prompts; multi-provider fallback (Gemma → DeepSeek → OpenAI proxy); 5 req/min rate limit.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lius-cc/lius-cookbook/blob/main/notebooks/how-we-build-llm-generate.ipynb)

📜 CC0 1.0

## 1 · Endpoint contract

```http
POST https://lius.cc/api/llm/generate
Content-Type: application/json

{
  "task": "narrative" | "publishing" | "grant",
  "topic": "主題 / 領域"
}
```

Returns:
```json
{
  "ok": true,
  "task": "...",
  "topic": "...",
  "output": "LLM 生成內容（含 [1] [2] 引用標號）",
  "provider": "gemma|deepseek|proxy",
  "hits": [{"name":"...","type":"...","url":"...","score":N}],
  "latency_ms": 11101,
  "snapshot": "2026-05-17"
}
```

## 2 · Live call

In [ ]:
import requests

r = requests.post(
    "https://lius.cc/api/llm/generate",
    json={"task": "narrative", "topic": "媽祖 海上守護"},
    timeout=60,
).json()

print(f"provider={r['provider']}  latency={r['latency_ms']}ms  hits={len(r['hits'])}")
print()
print(r['output'][:400])

## 3 · Three task prompts (curated)

**`narrative`** — 200-300 字文化敘事文案 (商品包裝、品牌故事、文化導覽)

**`publishing`** — 8-10 個出版企劃切角 + 目標讀者 + 一句話定位

**`grant`** — 200-250 字文化部標案級「文化考據段落」+ 引用清單

In [ ]:
# 從 single endpoint 拿三種輸出
tasks = [
    ("narrative",  "城隍 司法 善惡"),
    ("publishing", "中元 普渡 七月"),
    ("grant",      "在地宗教文化盤點"),
]

for task, topic in tasks:
    r = requests.post("https://lius.cc/api/llm/generate",
                      json={"task": task, "topic": topic}, timeout=60).json()
    if r.get("ok"):
        print(f"=== {task} · {topic} ({r['provider']}, {r['latency_ms']}ms) ===")
        print(r["output"][:300] + "...\n")
    else:
        print(f"=== {task} FAILED: {r.get('error')} ===\n")

## Reproducibility

- **Snapshot**: 2026-05-17
- **Rate limit**: 5 req/min per IP
- **Provider order**: Gemma 2.5 Flash → DeepSeek V4-Flash → OpenAI proxy (gpt-5.4-mini)
- **RAG**: top-5 hits with 1,200-char content snippets, scored by lius.cc/api/llm-rag scoring algorithm
- **Output**: text with inline `[1] [2]` citation markers tied to `hits[]`